# Combine Log Files By Date Folder

This notebook scans each month folder under `Processed_data/Audio` (for example `2026_02`), reads all experiment log text files inside that folder's experiment subdirectories, and writes one combined CSV back into the month folder.

Output columns include the experiment number, the start time parsed from the log filename, the timestamp for each log line, and the log text.


In [ ]:
import re
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("/mnt/home/gginosar/repos/gerbil_vocalization_analysis")
if (REPO_ROOT / "vocalization_analysis").exists() and str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

AUDIO_ROOT = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio")
MONTH_PATTERN = re.compile(r"^\d{4}_\d{2}$")
LOG_FILENAME_PATTERN = re.compile(
    r"experiment_(?P<exp>\d+)_log_(?P<start_time>\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})\.txt$"
)
LOG_LINE_PATTERN = re.compile(r"^(?P<log_time>[^>]+)>(?P<log_text>.*)$")

print("Audio root:", AUDIO_ROOT)


In [ ]:
def parse_log_file(log_path: Path) -> list[dict]:
    match = LOG_FILENAME_PATTERN.match(log_path.name)
    if match is None:
        raise ValueError(f"Unexpected log filename format: {log_path}")

    exp = int(match.group("exp"))
    experiment_start_time = pd.to_datetime(match.group("start_time"), format="%Y-%m-%d_%H-%M-%S")

    rows = []
    for line_number, raw_line in enumerate(log_path.read_text(errors="replace").splitlines(), start=1):
        line = raw_line.strip()
        if not line:
            continue
        if line.startswith("Time>Log for experiment number"):
            continue

        line_match = LOG_LINE_PATTERN.match(line)
        if line_match is None:
            rows.append(
                {
                    "month_folder": log_path.parents[1].name,
                    "exp": exp,
                    "experiment_start_time": experiment_start_time,
                    "log_time": pd.NaT,
                    "log_text": line,
                    "source_log_file": log_path.name,
                    "line_number": line_number,
                    "parse_status": "unparsed",
                }
            )
            continue

        rows.append(
            {
                "month_folder": log_path.parents[1].name,
                "exp": exp,
                "experiment_start_time": experiment_start_time,
                "log_time": pd.to_datetime(line_match.group("log_time"), format="%Y-%m-%d_%H-%M-%S", errors="coerce"),
                "log_text": line_match.group("log_text").strip(),
                "source_log_file": log_path.name,
                "line_number": line_number,
                "parse_status": "parsed",
            }
        )

    return rows


def build_month_log_csv(month_folder: Path) -> pd.DataFrame:
    log_paths = sorted(month_folder.glob("*/experiment_*_log*.txt"))
    combined_rows = []
    for log_path in log_paths:
        combined_rows.extend(parse_log_file(log_path))

    combined_df = pd.DataFrame(combined_rows)
    if combined_df.empty:
        return combined_df

    combined_df = combined_df.sort_values(
        by=["exp", "experiment_start_time", "log_time", "line_number"],
        kind="stable",
    ).reset_index(drop=True)
    return combined_df


In [ ]:
results = []

for month_folder in sorted(path for path in AUDIO_ROOT.iterdir() if path.is_dir() and MONTH_PATTERN.match(path.name)):
    print(f"Processing {month_folder.name}")
    combined_df = build_month_log_csv(month_folder)

    output_csv_path = month_folder / "combined_experiment_logs.csv"
    combined_df.to_csv(output_csv_path, index=False)

    print(f"  wrote {len(combined_df)} rows to {output_csv_path}")
    results.append(
        {
            "month_folder": month_folder.name,
            "num_rows": len(combined_df),
            "num_log_files": len(list(month_folder.glob('*/experiment_*_log*.txt'))),
            "output_csv": str(output_csv_path),
        }
    )

results_df = pd.DataFrame(results)
results_df


In [ ]:
import ast

RAW_ROOT = Path("/mnt/home/neurostatslab/ceph/saneslab_data/big_setup")

sync_results = []

for month_folder in sorted(path for path in AUDIO_ROOT.iterdir() if path.is_dir() and MONTH_PATTERN.match(path.name)):
    month_rows = []
    experiment_dirs = sorted(path for path in month_folder.iterdir() if path.is_dir() and path.name.isdigit())

    for experiment_dir in experiment_dirs:
        exp = int(experiment_dir.name)
        sync_path = RAW_ROOT / f"experiment_{exp}" / "concatenated_data_cam_mic_sync" / "sync.csv"
        raw_audio_folder = RAW_ROOT / f"experiment_{exp}" / "concatenated_data_cam_mic_sync"

        raw_file_numbers = set()
        for pattern in ("channel_00_file_*.wav", "channel_0_*.wav"):
            for wav_path in raw_audio_folder.glob(pattern):
                name = wav_path.name
                if "_file_" in name:
                    file_num_text = name.split("_file_")[1].split(".wav")[0]
                else:
                    file_num_text = name.split("channel_0_")[1].split(".wav")[0]
                try:
                    raw_file_numbers.add(int(file_num_text))
                except ValueError:
                    continue

        num_files = len(raw_file_numbers)

        if not sync_path.exists():
            month_rows.append(
                {
                    "exp": exp,
                    "start_time": pd.NaT,
                    "end_time": pd.NaT,
                    "num_files": num_files,
                    "num_hours": pd.NA,
                }
            )
            continue

        try:
            sync_df = pd.read_csv(sync_path)
            if "timestamp" not in sync_df.columns:
                month_rows.append(
                    {
                        "exp": exp,
                        "start_time": pd.NaT,
                        "end_time": pd.NaT,
                        "num_files": num_files,
                        "num_hours": pd.NA,
                    }
                )
                continue

            timestamps = sync_df["timestamp"].apply(ast.literal_eval)
            start_time = pd.to_datetime(timestamps.iloc[0][0])
            end_time = pd.to_datetime(timestamps.iloc[-1][-1])
            num_hours = (end_time - start_time).total_seconds() / 3600
            month_rows.append(
                {
                    "exp": exp,
                    "start_time": start_time,
                    "end_time": end_time,
                    "num_files": num_files,
                    "num_hours": num_hours,
                }
            )
        except Exception as exc:
            month_rows.append(
                {
                    "exp": exp,
                    "start_time": pd.NaT,
                    "end_time": pd.NaT,
                    "num_files": num_files,
                    "num_hours": pd.NA,
                }
            )

    month_df = pd.DataFrame(month_rows).sort_values("exp").reset_index(drop=True)
    output_csv_path = month_folder / "exp_times.csv"
    month_df.to_csv(output_csv_path, index=False)

    print(f"Processed {month_folder.name}: wrote {len(month_df)} rows to {output_csv_path}")
    sync_results.append(
        {
            "month_folder": month_folder.name,
            "num_experiments": len(month_df),
            "num_with_times": int(month_df["start_time"].notna().sum()) if not month_df.empty else 0,
            "output_csv": str(output_csv_path),
        }
    )

sync_results_df = pd.DataFrame(sync_results)
sync_results_df
